# The Billion-Dollar Eigenvector: Implementing and Analyzing the PageRank Algorithm

**A Technical Report on Link-Based Web Page Ranking**

---

## 1. Problem Formulation

### 1.1 Introduction & Motivation

In 1998, two Stanford graduate students, Larry Page and Sergey Brin, faced a significant challenge: how to organize the rapidly expanding World Wide Web [1]. At the time, the web contained hundreds of millions of pages with no central authority, no quality control, and no systematic way to distinguish authoritative sources from spam or self-promotion. Existing search engines relied primarily on keyword matching and rudimentary text analysis—methods easily manipulated by webmasters who could artificially inflate their pages' relevance through keyword stuffing [2].

Page and Brin's insight was revolutionary and elegant: the web's hyperlink structure itself encodes quality signals. When an author creates a hyperlink from their page to another, they are endorsing that destination as valuable or relevant. This is similar to academic citation networks, where influential papers are those cited frequently by other influential papers [1]. But translating this intuition into a computable algorithm required solving a mathematical paradox: *a page is important if important pages link to it—but which pages are important?*

Their solution, **PageRank**, transformed this definition into a tractable problem. By modeling web browsing as a random walk on a graph and finding the steady-state probability distribution, PageRank produces rankings that are:
- Objective: Derived from link structure, not human editors or self-declared importance.
- Stable: Robust to minor changes in the graph topology.  
- Scalable: Computable for billions of pages using iterative methods.
- Theoretically grounded: Based on fundamental principles from linear algebra and probability theory.

PageRank became the mathematical foundation of Google Search and remains one of the most famous applications of eigenvector centrality in computer science [3]. This report implements the original 1998 PageRank algorithm, analyzes its mathematical properties, and validates its convergence behavior through computational experiments.


### 1.2 The Web as a Graph

#### 1.2.1 Graph Representation

To apply mathematical analysis to the web, we first formalize its structure as a directed graph $G = (V, E)$, where:
- Nodes $V = \{p_1, p_2, \dots, p_n\}$ represent web pages.
- Directed edges $E \subseteq V \times V$ represent hyperlinks.

A directed edge $(p_j, p_i)$ exists if page $p_j$ contains a hyperlink to page $p_i$. Directionality is crucial: $(p_j, p_i) \in E$ does **not** imply $(p_i, p_j) \in E$ - linking is not symmetric [4].

A directed graph naturally captures the hyperlink structure of the web. Each node's outgoing edges (outlinks) represent the pages it endorses, while the incoming edges (inlinks or backlinks) represent endorsements it receives. This representation abstracts away content details and focuses purely on structural relationships - enabling mathematical analysis independent of language, topic, or multimedia content.

#### 1.2.2 The Link as a Vote

The core assumption of PageRank is that hyperlinks are "votes of confidence" [1]. When a website's author creates a link, they signal that the destination page is valuable, relevant, or authoritative within some context. This interpretation is justified by several observations:

1. Cost of creation: Adding a link requires deliberate action—authors link to resources they find genuinely useful
2. Editorial judgment: Unlike keywords (which anyone can add), links involve comparison and selection among alternatives
3. Network effects: Over time, the web self-organizes—high-quality resources accumulate links naturally through community consensus

However, not all votes are equal. A link from a widely-cited academic institution carries more weight than a link from an unknown personal blog. This intuition leads directly to the recursive definition of PageRank.

### 1.3 The Recursive Importance Concept

#### 1.3.1 The Circular Definition

PageRank formalizes web page importance through a recursive relationship:

> *A page is important if it is linked to by other important pages.*

Let's illustrate this with a simple 3-page web:

Example Graph:
```
Page A ──→ Page B ──→ Page C
  ↑                      │
  └──────────────────────┘
```

Intuitive reasoning:
- Page C is endorsed by Page B
- Page B is endorsed by Page A  
- Page A is endorsed by Page C

The paradox: To determine A's importance, we need to know C's importance. But C's importance depends on B's, which depends on A's. This circular dependency seems to prevent us from computing anything.

#### 1.3.2 Distributing Authority

The solution lies in distributing authority probabilistically. If Page B has 3 outgoing links, each link receives $\frac{1}{3}$ of B's "*voting power*". Mathematically, if page $p_j$ has importance score $r_j$ and $N_j$ outlinks, it contributes $\frac{r_j}{N_j}$ to each page it links to.

For our 3-page example (assuming each page has 1 outlink):
- $r_A$ receives contribution from $r_C$
- $r_B$ receives contribution from $r_A$  
- $r_C$ receives contribution from $r_B$

This gives us the system:
$$
\begin{align}
r_A &= r_C \\
r_B &= r_A \\
r_C &= r_B
\end{align}
$$

Subject to normalization: $r_A + r_B + r_C = 1$ (total probability sums to 1).

The solution is $r_A = r_B = r_C = \frac{1}{3}$ - all pages are equally important because the graph is symmetric.

This circular system of equations can be solved by finding the steady-state distribution of a random walk on the graph - which is precisely an eigenvector problem [2]. 

### 1.4 From Concept to Mathematics

#### 1.4.1 The Adjacency Matrix

We formalize the graph structure using an adjacency matrix $A \in \{0,1\}^{n \times n}$:

$$
A_{ij} = 
\begin{cases}
1 & \text{if page } j \text{ links to page } i \\
0 & \text{otherwise}
\end{cases}
$$

We use column-to-row indexing. If $A_{ij} = 1$, then column $j$ (source, origin) links to row $i$ (destination). This convention aligns with matrix-vector multiplication where columns represent distributions. [5]

Example (3-page graph above):
$$
A = \begin{bmatrix}
0 & 0 & 1 \\
1 & 0 & 0 \\
0 & 1 & 0
\end{bmatrix}
$$

Reading column 1 (Page A): A links to B (row 2).  
Reading column 2 (Page B): B links to C (row 3).  
Reading column 3 (Page C): C links to A (row 1).

#### 1.4.2 The Transition Matrix (Stochastic Matrix)

The adjacency matrix encodes topology but not probabilities. We transform it into a column-stochastic transition matrix $M$ by normalizing each column to sum to 1:

$$
M_{ij} = \frac{A_{ij}}{\sum_{k=1}^{n} A_{kj}} = \frac{A_{ij}}{N_j}
$$

where $N_j = |\{k : A_{kj} = 1\}|$ is the number of outlinks from page $j$ (the out-degree).

Interpretation: $M_{ij}$ is the probability that a random surfer on page $j$ clicks the link to page $i$.

For our example:
$$
M = \begin{bmatrix}
0 & 0 & 1 \\
1 & 0 & 0 \\
0 & 1 & 0
\end{bmatrix}
$$
(Each column already sums to 1, so $M = A$ in this case.)

Properties of $M$:
1. Column-stochastic: $\sum_{i=1}^{n} M_{ij} = 1$ for all $j$ (each column is a probability distribution).
2. Non-negative: $M_{ij} \geq 0$ (probabilities cannot be negative).
3. Directionality: Since links are directional, $M$ is generally not symmetric.

#### 1.4.3 The PageRank Vector

Let $\mathbf{r} = [r_1, r_2, \ldots, r_n]^T$ be the PageRank vector, where $r_i$ represents the importance score (or probability) of page $i$. The recursive importance relationship becomes:

$$
\mathbf{r} = M \mathbf{r}
$$

This is the eigenvector equation $M\mathbf{r} = \lambda \mathbf{r}$ with eigenvalue $\lambda = 1$. For any column-stochastic matrix, the dominant eigenvalue is always $\lambda = 1$ [6]. 

This is because:
- Stochastic matrices preserve probability (they map probability distributions to probability distributions).
- The sum of entries in $\mathbf{r}$ remains constant under multiplication by $M$.
- The Perron-Frobenius theorem guarantees a unique positive eigenvector for stochastic matrices. [7]

Physical interpretation: The PageRank vector $\mathbf{r}$ represents the steady-state probability distribution of an infinite random walk on the web graph. If a random surfer browses forever, $r_i$ is the long-term fraction of time spent on page $i$.[1]


### 1.5 Research Question

#### What was the challenge?

In 1998, search engines faced a fundamental problem: "how do you determine which pages are authoritative when anyone can publish anything?" With billions of web pages and no central authority, traditional approaches faced critical limitations:

- Computational scale: Human curation (like Yahoo's directory) couldn't keep pace with exponential web growth [8]
- Gaming vulnerability: Keyword-based ranking allowed webmasters to manipulate results through keyword stuffing.
- Objectivity: No systematic way to distinguish expert sources from self-promotion.

The identified opportunity was that the web's hyperlink structure itself contains implicit quality signals. A link is a recommendation — pages cite sources they find valuable. But how do we translate this intuition into computation?

#### Research Question

> How does the mathematical structure of the web's link network reveal page authority, and can we model this relationship to create an automated ranking system?

This question matters because:

- Stability: Rankings should reflect structural importance, not fluctuate with minor changes.
- Recursion paradox: A page is important if important pages link to it—but which pages are important? This circular definition requires mathematical resolution.
- Scalability: The solution must compute rankings for billions of pages in reasonable time.
- Mathematical approach: The steady-state distribution of a random surfer on the web graph turns out to be an *eigenvector*—connecting web search to fundamental linear algebra.

### 1.6 The Approach

To provide an answer to the main question above, we will go through two main phases: 
* Phase 1: Implementation.
* Phase 2: Experiments and Analysis.

#### **Phase 1: Implementation** 

##### Step 1.1: Model the Web as a Markov Chain

We will construct a directed graph of 5-7 web pages (including a spider trap for realism) and transform it into a column-stochastic transition matrix $M$. We will create a probability table. If Page B links to Pages A and C, column B shows 50% chance of jumping to A and 50% to C.

During this phase we will handle the below edge cases:
- Dangling nodes: Pages with no outgoing links (dead ends).
- Spider traps: Isolated cycles that trap random surfers (e.g., $A \leftrightarrow B$).
- Disconnected components: Graph clusters with no links between them.


##### Step 1.2: Implement Power Iteration for Steady-State

We will start with an initial guess $\mathbf{v}_0 = [\frac{1}{n}, \frac{1}{n}, \ldots, \frac{1}{n}]^T$ (uniform distribution) and repeatedly apply the transformation:

$$
\mathbf{v}_{k+1} = M \mathbf{v}_k
$$

until convergence (when $\|\mathbf{v}_{k+1} - \mathbf{v}_k\|_1 < \epsilon$, typically $\epsilon = 10^{-6}$).

In this way we will simulate the long-term behavior of a random surfer. After enough iterations, the probability distribution stabilizes—this is our ranking. When the vector stops changing, we've found the fixed point of the system—the steady-state where applying $M$ no longer alters the distribution. This represents the true structural importance of pages and provide the PageRank scores.

To ensure convergence and handle graph pathologies, we modify the transition matrix applying a damping factor:

$$
M' = d \cdot M + \frac{(1-d)}{n} \cdot \mathbf{E}
$$

where $d \in [0,1]$ (typically 0.85) and $\mathbf{E}$ is an $n \times n$ matrix of all ones. The term $\frac{(1-d)}{n}$ represents "teleportation"—the probability that a surfer randomly jumps to any page instead of following links [1].

##### Step 1.3: Validate Against Eigenvalue Decomposition

We will analytically compute all eigenvectors and eigenvalues of $M'$, then extract the eigenvector corresponding to $\lambda = 1$.

The steady-state of a Markov chain is mathematically defined as the eigenvector where $M'\mathbf{v} = 1 \cdot \mathbf{v}$ [9]. If our Power Iteration result matches the eigenvalue decomposition result (within numerical precision $\sim 10^{-4}$), we've proven:
- Our iteration converged to the correct eigenvector.
- Our transition matrix was constructed properly.  
- Our stopping criterion ($\epsilon = 10^{-6}$) is appropriate.

For a validation metric we are going to compute the absolute difference between the scoring derived from the two methods:

$$
\text{Error} = \|\mathbf{v}_{\text{iteration}} - \mathbf{v}_{\text{eig}}\|_1
$$

If Error $< 10^{-3}$, validation passes.


#### **Phase 2: Analysis & Experiments**

In this phase we are going to conduct four experiments to test PageRank's behavior, validate its mathematical properties and explore its practical applications.

##### Experiment 2.1: Initial Vector Independence

We are going to run Power Iteration from three different starting points (initial vectors) and test if all starting points converged to identical final PageRank scores: 
  1. Uniform: $\mathbf{v}_0 = [\frac{1}{n}, \ldots, \frac{1}{n}]^T$
  2. Concentrated: $\mathbf{v}_0 = [1, 0, \ldots, 0]^T$ (all probability on one page)
  3. Random: $\mathbf{v}_0$ sampled from Dirichlet distribution
- Show all converge to the same steady-state—proving objectivity
- Mathematical justification: The Perron-Frobenius theorem guarantees uniqueness [7]

##### Experiment 2.2: Personalized PageRank

We are going to test how biasing teleportation(random jumps) towards specific pages changes rankings. This will demonstrate how we can create personalized search based on personal history/preferences. 

- We will start by applying a standard uniform teleportation: (equal shares for all pages).
- Personalized Page: 100% teleportation to one single page
- Personalized Pages: 50% teleportations to spider trap pages.

##### Experiment 2.3: Network Connectivity

We are going to test what is the impact of removing strategic links on PageRank scores.

- We are going to remove a critical link between two page clusters in the graph and analyze the score change.
- Measure impact on rankings to demonstrate how web structure propagates the page authority.
- Measure sensitivity: $\Delta r_i = |r_i^{\text{before}} - r_i^{\text{after}}|$

##### Experiment 2.4: Link Manipulation (SEO Attack)

We are going to test how manipulative are the scores of the PageRank Algorithm by artificially adding links to low-ranking pages.

- We will get links from highly scored pages.
- We will creating fake Link Farms that add links from multiple pages to one page.
- Reduce link importance: We will test if lower lower damping factor actually minimizes the effect of link manipulations investigated in p.1 and p.2.


#### Expected Outcome

These experiments aims to validate the theoretical foundations and test the implementation from Phase 2, demonstrating that PageRank is:

1. **Mathematical soundness:**  The unique steady-state and guaranteed convergence from any initial vector empirically confirm the Perron-Frobenius theorem.[7]
2. **Flexible and practical:** Personalized teleportation enables context-aware ranking (e.g., user preferences) while preserving algorithmic rigor.
3. **Structurally dependent:** Link topology critically determines authority flow; removing or adding strategic links significantly reshapes rankings.
4. **Vulnerable to manipulation:** Pure PageRank can be gamed through link schemes, explaining why real search engines require hundreds of additional anti-spam signals beyond the core algorithm.

---

### References

[1] Page, L., Brin, S., Motwani, R., & Winograd, T. (1999). The PageRank Citation Ranking : Bringing Order to the Web. The Web Conference.

[2] Brin, S., & Page, L. (1998). *The Anatomy of a Large-Scale Hypertextual Web Search Engine*. Computer Networks and ISDN Systems, 30(1-7), 107-117. https://doi.org/10.1016/S0169-7552(98)00110-X

[3] Langville, A. N., & Meyer, C. D. (2011). Google’s PageRank and beyond. In *Princeton University Press eBooks*. https://doi.org/10.2307/j.ctt7t8z9

[4] Newman, M. E. J. (2018). *Networks* (2nd ed.). Oxford University Press. Chapter 7: Measures and Metrics.

[5] Strang, G. (2016). *Introduction to Linear Algebra* (5th ed.). Wellesley-Cambridge Press. Chapter 8: Linear Transformations.

[6] Meyer, C. D. (2000). *Matrix Analysis and Applied Linear Algebra*. SIAM. Chapter 8: The Perron-Frobenius Theory of Nonnegative Matrices.

[7] Seneta, E. (2006). *Non-negative Matrices and Markov Chains* (Revised Printing). Springer. Theorem 1.1: Perron-Frobenius Theorem.

[8] Battelle, J. (2005). *The Search: How Google and Its Rivals Rewrote the Rules of Business and Transformed Our Culture*. Portfolio. Chapter 4: The Birth of Google.

[9] Norris, J. R. (1998). *Markov Chains*. Cambridge University Press. Chapter 1: Stationary Distributions.

---